# Embedding 모델별 PaCMAP 군집 리포트

영화 plot summary 임베딩 결과(`result_data/result-{model}.csv`)를 온톨로지 라벨(`origin_data/movie_ontology.csv`)과 결합해,
모델별 **genre / theme(category) / mood** 군집 분리도를 2D·3D PaCMAP으로 시각화한 결과입니다.

> PaCMAP 입력은 cosine 거리와 맞추기 위해 L2 정규화한 임베딩을 사용합니다.

## 자료구조

| 파일 | 컬럼 |
|------|------|
| `origin_data/movie_summary_export.csv` | `movie_id`, `title`, `summary` |
| `origin_data/movie_ontology.csv` | `movie_id`, `title`, `genres[]`, `themes[]`, `moods[]` |
| `result_data/result-{model}.csv` | `title`, `genre`, `embedding_0` … `embedding_{d-1}` |

> 학습 CSV에는 `movie_id`가 없으므로 `title`로 `movie_summary_export`와 매핑한 뒤 `movie_id`를 부여합니다.
> `title` 중복이 있으면 해당 row는 제외합니다.

## 모델별 속도와 메모리 사용량
| 모델 | 메모리 사용량 | 속도 |
|------|------|------|
| BAAI/bge-m3 | 2.5 GB | 25item/s |
| Qwen/Qwen3-Embedding-0.6B | 1.5 GB | 10item/s |
| Qwen/Qwen3-Embedding-4B | 8 GB | 2.5item/s |


In [1]:
# 필요 시 의존성 설치 (최초 1회)
import importlib.util

_REQUIRED = ("numpy", "pandas", "pacmap", "matplotlib", "plotly", "sklearn")
if any(importlib.util.find_spec(m) is None for m in _REQUIRED):
    %pip install -q numpy pandas pacmap matplotlib plotly scikit-learn

In [2]:
from __future__ import annotations

import ast
import re
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
import pacmap

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["font.family"] = "AppleGothic" if Path("/System/Library/Fonts").exists() else "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "embedding":
    NOTEBOOK_DIR = Path("./")

ORIGIN_DIR = NOTEBOOK_DIR / "origin_data"
RESULT_DIR = NOTEBOOK_DIR / "result_data"

SUMMARY_CSV = ORIGIN_DIR / "movie_summary_export.csv"
ONTOLOGY_CSV = ORIGIN_DIR / "movie_ontology.csv"

LABEL_TYPES: dict[str, tuple[str, str]] = {
    "genre": ("genres", "label_genre"),
    "theme": ("themes", "label_theme"),  # ontology themes = category 축
    "mood": ("moods", "label_mood"),
}

PACMAP_CFG_2D = dict(n_components=2, n_neighbors=70, MN_ratio=0.5, FP_ratio=2.0, random_state=42)
PACMAP_CFG_3D = dict(n_components=3, n_neighbors=70, MN_ratio=0.5, FP_ratio=2.0, random_state=42)
SILHOUETTE_SAMPLE = 2000

In [3]:
def parse_ontology_csv(path: Path) -> pd.DataFrame:
    """리스트 필드에 쉼표가 포함된 ontology CSV를 정규식으로 파싱한다."""
    pattern = re.compile(r'^"([^"]+)", "([^"]*)", (\[.*\]), (\[.*\]), (\[.*\])$')
    rows: list[dict[str, Any]] = []
    bad_lines: list[int] = []

    for lineno, line in enumerate(path.read_text(encoding="utf-8").splitlines()[1:], start=2):
        match = pattern.match(line.strip())
        if not match:
            bad_lines.append(lineno)
            continue
        movie_id, title, genres_raw, themes_raw, moods_raw = match.groups()
        rows.append(
            {
                "movie_id": movie_id.strip(),
                "title": title.strip(),
                "genres": ast.literal_eval(genres_raw),
                "themes": ast.literal_eval(themes_raw),
                "moods": ast.literal_eval(moods_raw),
            }
        )

    if bad_lines:
        raise ValueError(f"ontology 파싱 실패 라인: {bad_lines[:5]} (총 {len(bad_lines)}건)")
    return pd.DataFrame(rows)


def primary_label(values: Any) -> str:
    if values is None or (isinstance(values, float) and np.isnan(values)):
        return "Unknown"
    if isinstance(values, (list, tuple)):
        cleaned = [str(v).strip() for v in values if str(v).strip()]
        return sorted(cleaned)[0] if cleaned else "Unknown"
    return str(values).strip() or "Unknown"


def load_summary(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    df["title"] = df["title"].astype(str).str.strip()
    return df


def report_duplicate_titles(summary: pd.DataFrame) -> tuple[set[str], pd.DataFrame]:
    dup_mask = summary["title"].duplicated(keep=False)
    dup_df = (
        summary.loc[dup_mask, ["title", "movie_id"]]
        .sort_values("title")
        .reset_index(drop=True)
    )
    dup_titles = set(dup_df["title"].unique())
    return dup_titles, dup_df


def load_result_csv(path: Path, dup_titles: set[str]) -> tuple[pd.DataFrame, int]:
    df = pd.read_csv(path)
    df["title"] = df["title"].astype(str).str.strip()
    dropped = int(df["title"].isin(dup_titles).sum())
    df = df.loc[~df["title"].isin(dup_titles)].copy()
    return df, dropped


def embedding_matrix(df: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    emb_cols = [c for c in df.columns if c.startswith("embedding_")]
    if not emb_cols:
        raise ValueError("embedding_* 컬럼이 없습니다.")
    emb_cols = sorted(emb_cols, key=lambda c: int(c.split("_")[1]))
    return df[emb_cols].to_numpy(dtype=float), emb_cols


def build_model_frame(
    result_df: pd.DataFrame,
    summary: pd.DataFrame,
    ontology: pd.DataFrame,
) -> pd.DataFrame:
    merged = result_df.merge(summary[["title", "movie_id"]], on="title", how="inner")
    merged = merged.merge(ontology, on="movie_id", how="left", suffixes=("", "_ont"))

    if "title_ont" in merged.columns:
        merged = merged.drop(columns=["title_ont"])

    for _, (src_col, label_col) in LABEL_TYPES.items():
        merged[label_col] = merged[src_col].apply(primary_label)

    return merged


def l2_normalize(X: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return X / norms


def run_pacmap(X: np.ndarray, n_components: int) -> np.ndarray:
    cfg = PACMAP_CFG_3D if n_components == 3 else PACMAP_CFG_2D
    reducer = pacmap.PaCMAP(**cfg)
    return reducer.fit_transform(l2_normalize(X), init="pca")


def clustering_metrics(X: np.ndarray, labels: pd.Series) -> dict[str, float | int | None]:
    y = pd.factorize(labels.astype(str))[0]
    n_classes = len(np.unique(y))
    n_samples = len(y)

    if n_classes < 2 or n_samples < n_classes + 1:
        return {
            "n_samples": n_samples,
            "n_classes": n_classes,
            "silhouette_cosine": None,
            "calinski_harabasz": None,
            "davies_bouldin": None,
        }

    sample_size = min(SILHOUETTE_SAMPLE, n_samples)
    sil = float(silhouette_score(X, y, metric="cosine", sample_size=sample_size, random_state=42))
    ch = float(calinski_harabasz_score(X, y))
    db = float(davies_bouldin_score(X, y))
    return {
        "n_samples": n_samples,
        "n_classes": n_classes,
        "silhouette_cosine": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
    }

In [4]:
summary_df = load_summary(SUMMARY_CSV)
ontology_df = parse_ontology_csv(ONTOLOGY_CSV)
dup_titles, dup_detail_df = report_duplicate_titles(summary_df)

print(f"summary 영화 수: {len(summary_df):,}")
print(f"ontology 영화 수: {len(ontology_df):,}")
print(f"중복 title 종류: {len(dup_titles):,}개")
print(f"중복 title 관련 row: {len(dup_detail_df):,}건 (결과 CSV에서 제외)")
display(dup_detail_df)

result_files = sorted(RESULT_DIR.glob("result-*.csv"))
if not result_files:
    raise FileNotFoundError(f"결과 CSV가 없습니다: {RESULT_DIR}")

model_frames: dict[str, pd.DataFrame] = {}
load_report_rows: list[dict[str, Any]] = []

for path in result_files:
    model_name = path.stem.replace("result-", "")
    raw_df, dropped = load_result_csv(path, dup_titles)
    frame = build_model_frame(raw_df, summary_df, ontology_df)
    X, emb_cols = embedding_matrix(frame)

    model_frames[model_name] = frame
    load_report_rows.append(
        {
            "model": model_name,
            "result_rows": len(raw_df) + dropped,
            "dropped_dup_title": dropped,
            "used_rows": len(frame),
            "embedding_dim": len(emb_cols),
            "ontology_matched": int(frame["genres"].notna().sum()),
            "unmapped_title": int(len(raw_df) - len(frame)),
        }
    )

load_report_df = pd.DataFrame(load_report_rows)
display(load_report_df)

summary 영화 수: 6,204
ontology 영화 수: 8,300
중복 title 종류: 29개
중복 title 관련 row: 58건 (결과 CSV에서 제외)


,title,movie_id
0,OCTONAUTS Season4,b5b5cb4f-c19e-41a8-87e4-7e09e0d8aa80
1,OCTONAUTS Season4,7a499794-e10c-4d69-b3f8-486dd4cc386a
2,君の膵臓をたべたい,cca36f27-abe2-4a08-9111-eadf4a843362
3,君の膵臓をたべたい,1d73a128-a432-4b02-9547-3f8ebe41f55d
4,弱虫ペダル,b236b0a8-a4d2-48cc-8eed-e3f97e50acb6
5,弱虫ペダル,17286d85-2cce-45c3-b66f-200c1fce1dcb
6,思い、思われ、ふり、ふられ,53bfe70f-18f7-4463-b66f-73214e8ca874
7,思い、思われ、ふり、ふられ,5f7667dc-c68e-45ba-be75-3f8fabefc271
8,고백,2a6a10e1-32a1-4a06-bdaa-c5cd8d15f2ba
9,고백,f9f660c9-08fb-42c4-8a52-90ddb234cad6


,model,result_rows,dropped_dup_title,used_rows,embedding_dim,ontology_matched,unmapped_title
0,bge-m3,6204,58,6146,1024,6146,0
1,qwen3-0_6b,6204,58,6146,1024,6146,0
2,qwen3-4b-1024,6204,58,6146,1024,6146,0
3,qwen3-4b-origin,6204,58,6146,2560,6146,0


In [5]:
def plot_matplotlib_2d(df: pd.DataFrame, label_col: str, title: str) -> None:
    fig, ax = plt.subplots(figsize=(11, 8))
    labels = df[label_col].astype(str)
    for name, group in df.groupby(labels, sort=False):
        ax.scatter(group["pacmap_x"], group["pacmap_y"], s=12, alpha=0.65, label=name)
    ax.set_xlabel("PaCMAP-1")
    ax.set_ylabel("PaCMAP-2")
    ax.set_title(title)
    if labels.nunique() <= 20:
        ax.legend(markerscale=2, fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()


def plot_matplotlib_3d(df: pd.DataFrame, label_col: str, title: str) -> None:
    fig = plt.figure(figsize=(11, 8))
    ax = fig.add_subplot(111, projection="3d")
    labels = df[label_col].astype(str)
    for name, group in df.groupby(labels, sort=False):
        ax.scatter(
            group["pacmap_x"],
            group["pacmap_y"],
            group["pacmap_z"],
            s=10,
            alpha=0.6,
            label=name,
        )
    ax.set_xlabel("PaCMAP-1")
    ax.set_ylabel("PaCMAP-2")
    ax.set_zlabel("PaCMAP-3")
    ax.set_title(title)
    if labels.nunique() <= 15:
        ax.legend(markerscale=2, fontsize=7, loc="best")
    plt.tight_layout()
    plt.show()


def plot_plotly_2d(df: pd.DataFrame, label_col: str, title: str) -> go.Figure:
    fig = px.scatter(
        df,
        x="pacmap_x",
        y="pacmap_y",
        color=label_col,
        hover_data=["title", "movie_id"],
        title=title,
        opacity=0.75,
        height=650,
    )
    fig.update_traces(marker=dict(size=6))
    fig.update_layout(legend_title_text=label_col)
    fig.show()
    return fig


def plot_plotly_3d(df: pd.DataFrame, label_col: str, title: str) -> go.Figure:
    fig = px.scatter_3d(
        df,
        x="pacmap_x",
        y="pacmap_y",
        z="pacmap_z",
        color=label_col,
        hover_data=["title", "movie_id"],
        title=title,
        opacity=0.8,
        height=700,
    )
    fig.update_traces(marker=dict(size=1))
    fig.update_layout(legend_title_text=label_col)
    fig.show()
    return fig

In [6]:
metric_rows: list[dict[str, Any]] = []
pacmap_cache: dict[str, dict[str, pd.DataFrame]] = {}

for model_name, frame in model_frames.items():
    X, _ = embedding_matrix(frame)
    coords_2d = run_pacmap(X, n_components=2)
    coords_3d = run_pacmap(X, n_components=3)

    vis_2d = frame.copy()
    vis_2d[["pacmap_x", "pacmap_y"]] = coords_2d

    vis_3d = frame.copy()
    vis_3d[["pacmap_x", "pacmap_y", "pacmap_z"]] = coords_3d

    pacmap_cache[model_name] = {"2d": vis_2d, "3d": vis_3d}

    for label_name, (_, label_col) in LABEL_TYPES.items():
        emb_metrics = clustering_metrics(X, frame[label_col])

        metric_rows.append(
            {
                "model": model_name,
                "label_type": label_name,
                "space": "embedding",
                **emb_metrics,
            }
        )

metrics_df = pd.DataFrame(metric_rows)
metrics_pivot = metrics_df.pivot_table(
    index=["model", "label_type"],
    columns="space",
    values="silhouette_cosine",
)

print("=== 모델 × 라벨별 Silhouette (cosine, 높을수록 군집 분리 우수) ===")
display(metrics_pivot.round(4))
display(metrics_df.sort_values(["model", "label_type", "space"]).reset_index(drop=True))

Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.


=== 모델 × 라벨별 Silhouette (cosine, 높을수록 군집 분리 우수) ===


space                       embedding
model           label_type           
bge-m3          genre         -0.1261
                mood          -0.0923
                theme         -0.0900
qwen3-0_6b      genre         -0.1543
                mood          -0.1088
                theme         -0.1653
qwen3-4b-1024   genre         -0.1967
                mood          -0.1656
                theme         -0.2027
qwen3-4b-origin genre         -0.1865
                mood          -0.1563
                theme         -0.1950

,model,label_type,space,n_samples,n_classes,silhouette_cosine,calinski_harabasz,davies_bouldin
0,bge-m3,genre,embedding,6146,44,-0.126121,6.261167,6.759893
1,bge-m3,mood,embedding,6146,33,-0.092292,6.214981,8.464211
2,bge-m3,theme,embedding,6146,50,-0.090023,4.079970,7.241513
3,qwen3-0_6b,genre,embedding,6146,44,-0.154267,16.077299,5.382138
4,qwen3-0_6b,mood,embedding,6146,33,-0.108807,16.398140,6.336011
5,qwen3-0_6b,theme,embedding,6146,50,-0.165272,9.894885,5.644608
6,qwen3-4b-1024,genre,embedding,6146,44,-0.196704,24.324819,4.791871
7,qwen3-4b-1024,mood,embedding,6146,33,-0.165642,21.751388,5.646167
8,qwen3-4b-1024,theme,embedding,6146,50,-0.202660,12.666447,5.051668
9,qwen3-4b-origin,genre,embedding,6146,44,-0.186518,23.407157,4.821125


## 리포트 해석 가이드

- **Silhouette (cosine)**: -1~1, 1에 가까울수록 동일 라벨끼리 응집·이질 라벨과 분리가 잘 됨. (원본 임베딩 공간 기준)
- **Calinski-Harabasz**: 값이 클수록 군집 간 분산 대비 군집 내 분산이 작음.
- **Davies-Bouldin**: 값이 작을수록 군집 품질이 좋음.
- **PaCMAP**: 전역·지역 구조를 함께 보존하는 차원 축소. 시각화 좌표는 L2 정규화 임베딩에 `n_neighbors=15`로 계산했습니다.
- 라벨은 배열의 **대표값(primary, 사전순 첫 항목)** 으로 색상을 지정했습니다.
- `theme` 축은 ontology의 `themes` 필드이며, 요청하신 category에 해당합니다.
- 중복 `title` row는 매핑 모호성을 피하기 위해 전체 파이프라인에서 제외했습니다.

In [7]:
for model_name in sorted(pacmap_cache):
    print("\n" + "=" * 80)
    print(f"MODEL: {model_name}")
    print("=" * 80)

    for label_name, (_, label_col) in LABEL_TYPES.items():
        vis_2d = pacmap_cache[model_name]["2d"]
        vis_3d = pacmap_cache[model_name]["3d"]

        title_base = f"{model_name} | {label_name}"

        # print(f"\n--- {label_name.upper()} | Matplotlib 2D ---")
        # plot_matplotlib_2d(vis_2d, label_col, f"{title_base} (PaCMAP 2D, matplotlib)")

        # print(f"--- {label_name.upper()} | Matplotlib 3D ---")
        # plot_matplotlib_3d(vis_3d, label_col, f"{title_base} (PaCMAP 3D, matplotlib)")

        # print(f"--- {label_name.upper()} | Plotly 2D ---")
        # plot_plotly_2d(vis_2d, label_col, f"{title_base} (PaCMAP 2D, plotly)")

        print(f"--- {label_name.upper()} | Plotly 3D ---")
        plot_plotly_3d(vis_3d, label_col, f"{title_base} (PaCMAP 3D, plotly)")


MODEL: bge-m3
--- GENRE | Plotly 3D ---


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed